In [1]:
!pip install datasets
!pip install transformers torch sentencepiece

In [36]:
from google.colab import userdata
import os

from huggingface_hub import whoami, logout, create_repo, upload_folder

try:
    logout()
    print("Mevcut Hugging Face oturumu kapatıldı.")
except Exception as e:
    print(f"Logout sırasında bir durum oluştu: {e}")

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

try:
    user_info = whoami()
    print(f"Giriş yapılan kullanıcı: {user_info['name']}")
    print(f"E-posta: {user_info['email']}")
    print(f"Yetki seviyesi: {user_info['auth']['type']}")
except Exception as e:
    print("Giriş yapılamadı veya token geçersiz. Lütfen token'ı kontrol edin.")

Mevcut Hugging Face oturumu kapatıldı.
Giriş yapılan kullanıcı: aliemre2023
E-posta: aliemrekaya239@gmail.com
Yetki seviyesi: access_token


In [18]:
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
import torch
from transformers import (
    pipeline,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [7]:
device = 0 if torch.cuda.is_available() else -1
print(device)

0


In [8]:
ds = load_dataset("mltrev23/financial-sentiment-analysis")

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


archive.zip:   0%|          | 0.00/282k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5842 [00:00<?, ? examples/s]

In [9]:
print(ds)
print(ds["train"].column_names)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['Sentence', 'Sentiment'],
        num_rows: 5842
    })
})
['Sentence', 'Sentiment']
{'Sentence': "The GeoSolutions technology will leverage Benefon 's GPS solutions by providing Location Based Search Technology , a Communities Platform , location relevant multimedia content and a new and powerful commercial model .", 'Sentiment': 'positive'}


In [10]:
df = ds["train"].to_pandas()

data = df[["Sentence", "Sentiment"]]
data.head(5)

,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,According to the Finnish-Russian Chamber of Co...,neutral
4,The Swedish buyout firm has sold its remaining...,neutral


In [11]:
model_id = "facebook/nllb-200-distilled-600M"

print("Model ve Tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

Model ve Tokenizer yükleniyor...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

In [13]:
def translate_batch(text_list, src_lang="eng_Latn", tgt_lang="tur_Latn", batch_size=64):
    translated_texts = []

    # Kaynak dili
    tokenizer.src_lang = src_lang

    for i in range(0, len(text_list), batch_size):
        batch = text_list[i : i + batch_size]

        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                # Hedef dil
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
                max_length=512
            )

        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        translated_texts.extend(decoded)

        if (i // batch_size) % 5 == 0:
            print(f"İlerleme: {i}/{len(text_list)} satır tamamlandı...")

    return translated_texts

In [16]:
sentences = data["Sentence"].tolist()
print(sentences[0])

The GeoSolutions technology will leverage Benefon 's GPS solutions by providing Location Based Search Technology , a Communities Platform , location relevant multimedia content and a new and powerful commercial model .


In [17]:
print("Çeviri başladı...")
data["Translated_Sentence"] = translate_batch(sentences, batch_size=128)
print("Çeviri bitti...")

Çeviri başladı...
İlerleme: 0/5842 satır tamamlandı...
İlerleme: 640/5842 satır tamamlandı...
İlerleme: 1280/5842 satır tamamlandı...
İlerleme: 1920/5842 satır tamamlandı...
İlerleme: 2560/5842 satır tamamlandı...
İlerleme: 3200/5842 satır tamamlandı...
İlerleme: 3840/5842 satır tamamlandı...
İlerleme: 4480/5842 satır tamamlandı...
İlerleme: 5120/5842 satır tamamlandı...
İlerleme: 5760/5842 satır tamamlandı...
Çeviri bitti...


In [19]:
data.to_csv("translated_data.csv", index=False)
print("Dosya kaydedildi.")

Dosya kaydedildi.


In [20]:
data.head()

,Sentence,Sentiment,Translated_Sentence
0,The GeoSolutions technology will leverage Bene...,positive,"GeoSolutions teknolojisi , Benefon ' un GPS çö..."
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative,"BİR düşük seviyede, aşağı $1.50'den $2.50'e ka..."
2,"For the last quarter of 2010 , Componenta 's n...",positive,2010 yılının son çeyreğinde Componenta ' nın n...
3,According to the Finnish-Russian Chamber of Co...,neutral,"Finlandiya-Rusya Ticaret Odası ' na göre , Fin..."
4,The Swedish buyout firm has sold its remaining...,neutral,"İsveçli satın alma firması , şirketin Finlandi..."


In [21]:
label_map = {"negative": 0, "neutral": 1, "positive": 2}
data['label'] = data['Sentiment'].map(label_map)

In [22]:
data.head()

,Sentence,Sentiment,Translated_Sentence,label
0,The GeoSolutions technology will leverage Bene...,positive,"GeoSolutions teknolojisi , Benefon ' un GPS çö...",2
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative,"BİR düşük seviyede, aşağı $1.50'den $2.50'e ka...",0
2,"For the last quarter of 2010 , Componenta 's n...",positive,2010 yılının son çeyreğinde Componenta ' nın n...,2
3,According to the Finnish-Russian Chamber of Co...,neutral,"Finlandiya-Rusya Ticaret Odası ' na göre , Fin...",1
4,The Swedish buyout firm has sold its remaining...,neutral,"İsveçli satın alma firması , şirketin Finlandi...",1


In [27]:
train_df, test_df = train_test_split(data, test_size=0.2, random_state=42, stratify=data['label'])
print(f"Eğitim seti boyutu: {len(train_df)} satır")
print(f"Test seti boyutu: {len(test_df)} satır")

print("\nEğitim Seti Duygu Dağılımı:")
print(train_df['label'].value_counts(normalize=True))

print("\nTest Seti Duygu Dağılımı:")
print(test_df['label'].value_counts(normalize=True))

Eğitim seti boyutu: 4673 satır
Test seti boyutu: 1169 satır

Eğitim Seti Duygu Dağılımı:
label
1    0.535844
2    0.316927
0    0.147229
Name: proportion, dtype: float64

Test Seti Duygu Dağılımı:
label
1    0.535500
2    0.317365
0    0.147134
Name: proportion, dtype: float64


In [28]:
train_dataset = Dataset.from_pandas(train_df[['Translated_Sentence', 'label']])
test_dataset = Dataset.from_pandas(test_df[['Translated_Sentence', 'label']])

In [29]:
model_id = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [30]:
def tokenize_function(examples):
    return tokenizer(examples["Translated_Sentence"], padding="max_length", truncation=True)

In [31]:
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/4673 [00:00<?, ? examples/s]

Map:   0%|          | 0/1169 [00:00<?, ? examples/s]

In [32]:
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=3)

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
def compute_metrics(p):
    predictions = np.argmax(p.predictions, axis=1)
    accuracy = accuracy_score(p.label_ids, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, predictions, average='weighted')
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [34]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.532332,0.763045,0.749485,0.763045,0.753513
2,0.619576,0.523920,0.776732,0.756657,0.776732,0.759672
3,0.619576,0.557264,0.751925,0.758560,0.751925,0.754813


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=879, training_loss=0.49932930765162825, metrics={'train_runtime': 310.3912, 'train_samples_per_second': 45.166, 'train_steps_per_second': 2.832, 'total_flos': 3688587003128832.0, 'train_loss': 0.49932930765162825, 'epoch': 3.0})

In [37]:
model_name = "berturk-financial-sentiment-analysis"
user_name = "aliemre2023"
repo_id = f"{user_name}/{model_name}"

trainer.save_model(f"./{model_name}")
tokenizer.save_pretrained(f"./{model_name}")
print("Model klasöre kaydedildi.")

# Ensure the repository exists on Hugging Face Hub
# Set private=True if you want the repository to be private
create_repo(repo_id, repo_type="model", exist_ok=True, private=False)

# Upload model and tokenizer files directly to the Hugging Face Hub
# Use create_pr=True to ensure it works even if direct push is restricted
upload_folder(
    repo_id=repo_id,
    folder_path=f"./{model_name}",
    commit_message="Add fine-tuned model and tokenizer",
    create_pr=True
)

print("Model huggingfacee kaydedildi.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model klasöre kaydedildi.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...nalysis/model.safetensors:   0%|          |  556kB /  443MB            

  ...nalysis/training_args.bin: 100%|##########| 5.14kB / 5.14kB            

Model huggingfacee kaydedildi.


In [38]:
dataset_name = "turkish-financial-sentiment-dataset"
user_name = "aliemre2023"

hf_dataset = Dataset.from_pandas(data)
hf_dataset.push_to_hub(f"{user_name}/{dataset_name}", create_pr=True)
print("Data huggingfacee kaydedildi.")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  962kB /  962kB            

Data huggingfacee kaydedildi.


In [40]:
# Yeni cümleleri Dataset formatına çevirelim
test_texts = [
    "Şirket hisseleri geri alım programı sonrası tavan fiyata ulaştı.",
    "Global piyasalardaki resesyon endişeleri ihracat rakamlarını baskılıyor.",
    "Olağan genel kurul toplantısı haftaya pazartesi yapılacak."
]

# Tokenize işlemi
new_test_ds = Dataset.from_dict({"Translated_Sentence": test_texts})
tokenized_new = new_test_ds.map(lambda x: tokenizer(x["Translated_Sentence"], truncation=True, padding="max_length"), batched=True)

# Tahminleri al
predictions = trainer.predict(tokenized_new)
preds = np.argmax(predictions.predictions, axis=-1)

# Sonuçları eşleştir (0: Neg, 1: Nötr, 2: Poz)
label_map_rev = {0: "Negative", 1: "Neutral", 2: "Positive"}
for text, p in zip(test_texts, preds):
    print(f"{text} --> {label_map_rev[p]}")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Şirket hisseleri geri alım programı sonrası tavan fiyata ulaştı. --> Positive
Global piyasalardaki resesyon endişeleri ihracat rakamlarını baskılıyor. --> Negative
Olağan genel kurul toplantısı haftaya pazartesi yapılacak. --> Neutral
